In [ ]:
# Back up database cũ
from pathlib import Path
import shutil

db_path = Path("../data/database/career.db")
backup_path = db_path.with_name("career_backup.db")

shutil.copy2(db_path, backup_path)

print("Backup created:", backup_path.resolve())

Backup created: D:\Hoang's Project\AI-Job-Impact-Analyzer\data\database\career_backup.db


In [1]:
import pandas as pd
import pathlib as Path


work_activities_path = "../data/raw/onet/work_activities.csv"

df = pd.read_csv(work_activities_path)

df.head()

df["Scale ID"].value_counts()


Scale ID
IM    37351
LV    37351
Name: count, dtype: int64

In [2]:
required_columns = [
    "O*NET-SOC Code",
    "Element Name",
    "Scale ID",
    "Data Value"
]

missing_columns = set(required_columns) - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

print("Scale IDs:", df["Scale ID"].value_counts().to_dict())
print("Number of occupations:", df["O*NET-SOC Code"].nunique())
print("Number of activities:", df["Element Name"].nunique())

Scale IDs: {'IM': 37351, 'LV': 37351}
Number of occupations: 911
Number of activities: 41


# 1. Chuyển dữ liệu từ dạng dài sang dạng rộng

Dữ liệu gốc:

| Occupation | Activity | Scale ID | Data Value |
|------------|----------|----------|------------|
| Nghề A | Analyzing Data | IM | 4.5 |
| Nghề A | Analyzing Data | LV | 4.0 |
| Nghề A | Working with Computers | IM | 4.8 |
| Nghề A | Working with Computers | LV | 4.5 |

Sau khi pivot theo `Scale ID`:

| Occupation | Activity | Importance (IM) | Level (LV) |
|------------|----------|-----------------|------------|
| Nghề A | Analyzing Data | 4.5 | 4.0 |
| Nghề A | Working with Computers | 4.8 | 4.5 |


In [3]:
df["Scale ID"].nunique()

2

In [4]:
# Chọn các cột cần sử dụng
df = df[
    [
        "O*NET-SOC Code",
        "Element Name",
        "Scale ID",
        "Data Value"
    ]
].copy()

# Đổi tên cột để thuận tiện xử lý
df = df.rename(
    columns={
        "O*NET-SOC Code": "occupation_code",
        "Element Name": "activity",
        "Scale ID": "scale_id",
        "Data Value": "score"
    }
)

# Kiểm tra bản ghi trùng trước khi pivot
duplicate_mask = df.duplicated(
    subset=["occupation_code", "activity", "scale_id"],
    keep=False
)

print("Duplicate source records:", duplicate_mask.sum())



Duplicate source records: 0


In [5]:
if duplicate_mask.any():
    raise ValueError(
        "Duplicate occupation/activity/scale records found. "
        "Inspect them before pivoting."
    )

work_activities = (
    df.pivot(
        index=["occupation_code", "activity"],
        columns="scale_id",
        values="score"
    )
    .reset_index()
)

work_activities.columns.name = None

work_activities = work_activities.rename(
    columns={
        "IM": "importance_score",
        "LV": "level_score"
    }
)

print(work_activities.shape)
print(work_activities.head())

(37351, 4)
  occupation_code                                           activity  \
0      11-1011.00                      Analyzing Data or Information   
1      11-1011.00                    Assisting and Caring for Others   
2      11-1011.00                     Coaching and Developing Others   
3      11-1011.00  Communicating with People Outside the Organiza...   
4      11-1011.00  Communicating with Supervisors, Peers, or Subo...   

   importance_score  level_score  
0              4.38         4.84  
1              3.31         3.10  
2              4.71         5.85  
3              4.55         5.47  
4              4.75         5.82  


## Tạo bảng mới SQLite

In [6]:
import sqlite3

db_path = ("../data/database/career.db")

conn = sqlite3.connect(db_path)



In [ ]:


conn.execute("""
    CREATE TABLE IF NOT EXISTS occupation_work_activities (
        occupation_code TEXT NOT NULL,
        activity TEXT NOT NULL,
        importance_score REAL,
        level_score REAL,

        PRIMARY KEY (occupation_code, activity),

        FOREIGN KEY (occupation_code)
            REFERENCES occupations(occupation_code)
    )
""")

conn.commit()

In [8]:
# Kiểm tra mã nghề trước insert
occupation_codes = pd.read_sql_query(
    "SELECT occupation_code FROM occupations",
    conn
)

valid_codes = set(occupation_codes["occupation_code"])

source_codes = set(work_activities["occupation_code"])

matched_codes = source_codes & valid_codes
unmatched_codes = source_codes - valid_codes

print("Source occupations:", len(source_codes))
print("Matched occupations:", len(matched_codes))
print("Unmatched occupations:", len(unmatched_codes))



Source occupations: 911
Matched occupations: 911
Unmatched occupations: 0


In [9]:
work_activities_db = work_activities[
    work_activities["occupation_code"].isin(valid_codes)
].copy()

print("Rows to insert:", len(work_activities_db))

Rows to insert: 37351


In [10]:
# 1. Kiểm tra tổng số dòng và số nghề
print("Total rows:", len(work_activities_db))
print("Unique occupations:", work_activities_db["occupation_code"].nunique())
print("Unique activities:", work_activities_db["activity"].nunique())

# 2. Kiểm tra số hoạt động trên mỗi nghề
activity_counts = work_activities_db.groupby("occupation_code").size()

print("\nActivity count distribution:")
print(activity_counts.value_counts().sort_index())

# 3. Kiểm tra giá trị thiếu
print("\nMissing values:")
print(
    work_activities_db[
        ["importance_score", "level_score"]
    ].isna().sum()
)

# 4. Kiểm tra trùng khóa
print(
    "\nDuplicate occupation-activity pairs:",
    work_activities_db.duplicated(
        subset=["occupation_code", "activity"]
    ).sum()
)

Total rows: 37351
Unique occupations: 911
Unique activities: 41

Activity count distribution:
41    911
Name: count, dtype: int64

Missing values:
importance_score    0
level_score         0
dtype: int64

Duplicate occupation-activity pairs: 0


In [11]:
records = work_activities_db[
    [
        "occupation_code",
        "activity",
        "importance_score",
        "level_score"
    ]
].copy()

# Chuyển NaN thành None để SQLite lưu thành NULL
records = records.astype(object).where(
    pd.notna(records),
    None
)

try:
    with conn:
        conn.execute(
            "DELETE FROM occupation_work_activities"
        )

        conn.executemany(
            """
            INSERT INTO occupation_work_activities (
                occupation_code,
                activity,
                importance_score,
                level_score
            )
            VALUES (?, ?, ?, ?)
            """,
            records.itertuples(index=False, name=None)
        )

    print("Inserted rows:", len(records))

except Exception:
    raise

finally:
    conn.close()

Inserted rows: 37351


In [12]:
conn = sqlite3.connect(db_path)

In [13]:
pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT occupation_code) AS total_occupations,
        COUNT(DISTINCT activity) AS total_activities
    FROM occupation_work_activities
    """,
    conn
)

,total_rows,total_occupations,total_activities
0,37351,911,41
